In [ ]:
import requests
import keyring
import urllib3
from pyodata import Client

# ===================== 🔐 凭据管理补充 =====================
# 第一次运行此脚本前，请取消下面两行的注释以存入密码，运行一次后即可重新注释掉
# pwd_to_save = "你的SAP密码"
# keyring.set_password("sap_100", "sap user name", pwd_to_save)
# =========================================================

# 1. 基础配置
SERVICE_URL = 'https://yousap/sap/opu/odata/sap/ZMTEST5_UMD_SRV/'
SAP_USER = "sap user name"
SERVICE_NAME = "sap_100"

# 关闭证书警告
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# 2. 安全获取密码
pwd = keyring.get_password(SERVICE_NAME, SAP_USER)

if not pwd:
    raise ValueError(f"未在系统中找到服务 '{SERVICE_NAME}' 用户 '{SAP_USER}' 的密码。请先使用 keyring.set_password 存储密码。")

# 3. 创建会话
session = requests.Session()
session.auth = (SAP_USER, pwd)
session.verify = False  # 开发环境常用，生产环境建议配置证书
session.timeout = 10

try:
    # 1. 初始化客户端
    sap_client = Client(SERVICE_URL, session)
    
    # 2. 获取实体集 (scarrSet 是 SAP 标准航司示例表)
    scarr_set = sap_client.entity_sets.scarrSet

    # 3. 执行查询
    scarr_entities = scarr_set.get_entities().execute()
    
    # ===================== 🔥 灵活输出部分 =====================
    fields = [
        ("Airline Code", "Carrid"),
        ("Airline name", "Carrname"),
        ("Local currency", "Currcode"),
        ("Airline URL", "Url")
    ]

    count = len(scarr_entities)
    print(f"=== 灵活输出字段 (共找到 {count} 条记录) ===\n")
    
    if count == 0:
        print("未找到匹配的数据。")
    else:
        for scarr_entity in scarr_entities:
            print("-" * 40)
            for label, field_name in fields:
                # getattr 允许我们动态访问属性， field_name 是字符串
                value = getattr(scarr_entity, field_name, "N/A") 
                print(f"{label:25}: {value}")

except Exception as e:
    print(f"发生错误: {e}")